## 🧠 U-Net Model Architecture

U-Net is the state-of-the-art architecture for medical image segmentation.

### Architecture Overview:
- **Encoder (Downsampling):** Extract features, reduce spatial dimensions
- **Bottleneck:** Deepest layer with most features
- **Decoder (Upsampling):** Reconstruct segmentation map
- **Skip Connections:** Preserve spatial information

In [5]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Libraries imported!")
print(f"🎯 Device: {device}")
if torch.cuda.is_available():
    print(f"💪 GPU: {torch.cuda.get_device_name(0)}")


✅ Libraries imported!
🎯 Device: cuda
💪 GPU: NVIDIA GeForce GTX 1650


## 🧱 Double Convolution Block

In [3]:
class DoubleConv(nn.Module):
    """
    Double Convolution Block: Conv -> ReLU -> Conv -> ReLU
    Used in both encoder and decoder
    """
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

print("✅ DoubleConv block defined!")
print("   Structure: Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU")


✅ DoubleConv block defined!
   Structure: Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU


## 🏗️ U-Net Model Class## 

In [7]:
class UNet(nn.Module):
    """U-Net Architecture for Brain Tumor Segmentation"""
    def __init__(self, in_channels=4, out_channels=3):
        super(UNet, self).__init__()
        
        # Encoder (Downsampling)
        self.encoder1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        
        self.encoder2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        self.encoder3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        
        self.encoder4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)
        
        # Decoder (Upsampling)
        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.decoder4 = DoubleConv(1024, 512)
        
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = DoubleConv(512, 256)
        
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = DoubleConv(256, 128)
        
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = DoubleConv(128, 64)
        
        # Output layer
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        # Encoder with skip connections
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))
        
        # Bottleneck
        bottleneck = self.bottleneck(self.pool4(enc4))
        
        # Decoder with skip connections
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat([dec4, enc4], dim=1)
        dec4 = self.decoder4(dec4)
        
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.decoder1(dec1)
        
        # Output
        out = self.out(dec1)
        return torch.sigmoid(out)

print("✅ U-Net model class defined!")

✅ U-Net model class defined!


## 📊 Initialize Model

In [8]:
# Initialize model and move to GPU
model = UNet(in_channels=4, out_channels=3).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✅ U-Net model initialized!")
print(f"🎯 Device: {device}")
print(f"\n📊 Model Statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / (1024**2):.2f} MB")


✅ U-Net model initialized!
🎯 Device: cuda

📊 Model Statistics:
   Total parameters: 31,044,227
   Trainable parameters: 31,044,227
   Model size: ~118.42 MB


## 🧪 Test Forward Pass

In [9]:
# Test model with dummy input
test_input = torch.randn(2, 4, 240, 240).to(device)  # Batch of 2
test_output = model(test_input)

print("🧪 Forward pass test:")
print(f"   Input shape: {test_input.shape}")
print(f"   Output shape: {test_output.shape}")
print(f"   Output range: [{test_output.min():.3f}, {test_output.max():.3f}]")
print(f"\n✅ Model working correctly!")
print(f"🔥 Ready for loss functions and training!")


🧪 Forward pass test:
   Input shape: torch.Size([2, 4, 240, 240])
   Output shape: torch.Size([2, 3, 240, 240])
   Output range: [0.133, 0.881]

✅ Model working correctly!
🔥 Ready for loss functions and training!


## ✅ Model Architecture Complete!

### Summary:
- ✅ U-Net architecture implemented
- ✅ 31 Million parameters (~118 MB)
- ✅ GPU ready: NVIDIA GeForce GTX 1650
- ✅ Forward pass tested successfully
- ✅ Input: (4, 240, 240) - 4 MRI modalities
- ✅ Output: (3, 240, 240) - WT, TC, ET masks

### Model Structure:
- **Encoder:** 4 levels (64 → 128 → 256 → 512)
- **Bottleneck:** 1024 channels
- **Decoder:** 4 levels (512 → 256 → 128 → 64)
- **Skip Connections:** Encoder ↔ Decoder
- **Output:** Sigmoid activation for binary masks

### Next Steps:
1. **Loss Functions** - Dice Loss + Binary Cross Entropy
2. **Training Loop** - Forward/Backward pass
3. **Model Training** - Train on GPU
4. **Evaluation** - Validation metrics